In [1]:
# 02_sacp_proxy_and_override.ipynb -- quantitative SACP proxy with prediction intervals; statistically-screened override subset

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Dataframe I/O that needs no parquet engine: use Parquet if one is installed, else CSV.
def _parquet_ok():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod); return True
        except Exception:
            pass
    return False
PARQUET = _parquet_ok()
def write_df(df: pd.DataFrame, stem: Path):
    # 'stem' is a path without extension; returns the path actually written.
    if PARQUET:
        p = stem.with_suffix(".parquet"); df.to_parquet(p); return p
    p = stem.with_suffix(".csv"); df.to_csv(p, index=False); return p
def df_exists(stem: Path) -> bool:
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()
def read_df(stem: Path) -> pd.DataFrame:
    pq, cv = stem.with_suffix(".parquet"), stem.with_suffix(".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(f"{stem}.parquet/.csv")

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 02: quantitative-only SACP proxy with a predictive interval; an override is a rating that
#         falls OUTSIDE the band AND clears a notch floor, separating genuine qualitative
#         overrides from ordinary model error (the weakness seen in the first pilot) ---
THRESH = 2            # minimum |notch gap| to qualify as an override (floor)
LO, HI = 0.10, 0.90  # central 80% predictive interval from quantile models
src = PROC / "corpus"
if not df_exists(src):
    print("[skip] run 01_data_construction first (missing data/processed/corpus).")
else:
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.model_selection import cross_val_predict, KFold
    df = read_df(src)
    feat = json.loads((PROC / "feature_columns.json").read_text())
    X = df[feat].to_numpy()
    y = df["rating_rank"].to_numpy().astype(float)
    cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

    def oof(quantile=None):
        # Out-of-fold predictions; quantile=None gives the point (squared-error) proxy.
        if quantile is None:
            m = HistGradientBoostingRegressor(random_state=SEED)
        else:
            m = HistGradientBoostingRegressor(loss="quantile", quantile=quantile,
                                              random_state=SEED)
        return cross_val_predict(m, X, y, cv=cv)

    pred = oof(None)                 # point proxy = the standalone quantitative rank
    pred_lo, pred_hi = oof(LO), oof(HI)
    pred_lo = np.minimum(pred_lo, pred)   # enforce lo <= point <= hi defensively
    pred_hi = np.maximum(pred_hi, pred)

    clip = lambda a: np.clip(np.rint(a), 0, len(RATING_SCALE) - 1).astype(int)
    df["sacp_proxy_rank"] = clip(pred)
    df["sacp_low"], df["sacp_high"] = clip(pred_lo), clip(pred_hi)
    df["delta"] = (df["rating_rank"] - df["sacp_proxy_rank"]).astype(int)

    # Robust error scale from out-of-fold residuals (MAD -> sigma); z-score of the gap.
    resid = y - pred
    mad = np.median(np.abs(resid - np.median(resid)))
    sigma = float(1.4826 * mad) if mad > 0 else float(resid.std() or 1.0)
    df["delta_z"] = (df["rating_rank"] - pred) / sigma

    # Override = rating OUTSIDE the predictive band AND |delta| >= THRESH notches.
    outside_band = (df["rating_rank"] < df["sacp_low"]) | (df["rating_rank"] > df["sacp_high"])
    df["is_override"] = outside_band & (df["delta"].abs() >= THRESH)
    df["override_dir"] = np.where(df["delta"] > 0, "up",
                                  np.where(df["delta"] < 0, "down", "none"))

    write_df(df, PROC / "corpus_scored")
    ov = df[df["is_override"]].copy()
    write_df(ov, PROC / "override_subset")

    save_table(df["delta"].value_counts().sort_index().rename_axis("delta").reset_index(name="n"),
               "02_delta_distribution")
    coverage = float(((df["rating_rank"] >= df["sacp_low"]) &
                      (df["rating_rank"] <= df["sacp_high"])).mean())
    save_table(pd.DataFrame({
        "metric": ["n_firms", "override_n", "override_rate", "mean_abs_delta",
                   "resid_sigma_notches", "band_coverage"],
        "value": [len(df), int(ov.shape[0]), round(ov.shape[0] / len(df), 3),
                  round(float(df["delta"].abs().mean()), 3), round(sigma, 3),
                  round(coverage, 3)]}), "02_proxy_summary")

    # Figure (a): notch-gap histogram with the override floor marked (grayscale, 600 dpi, PNG+PDF).
    fig, ax = plt.subplots(figsize=(3.4, 2.6))
    sns.histplot(df["delta"], bins=range(int(df["delta"].min()), int(df["delta"].max()) + 2),
                 color="0.6", edgecolor="0.2", ax=ax)
    ax.axvline(0, color="0.1", lw=0.8)
    for t in (-THRESH, THRESH):
        ax.axvline(t, color="0.1", lw=0.8, ls="--")
    ax.set_xlabel(r"Notch gap $\Delta$ = agency - quantitative proxy")
    ax.set_ylabel("Firms")
    sns.despine(ax=ax)
    save_fig(fig, "02_delta_histogram")

    # Figure (b): override firms by direction (grayscale bars).
    dd = (ov["override_dir"].value_counts().reindex(["up", "down"]).fillna(0)
          .rename_axis("direction").reset_index(name="n"))
    fig, ax = plt.subplots(figsize=(3.0, 2.6))
    ax.bar(dd["direction"], dd["n"], color=["0.35", "0.7"], edgecolor="0.2", width=0.6)
    ax.set_ylabel("Override firms")
    ax.set_xlabel("Direction")
    sns.despine(ax=ax)
    save_fig(fig, "02_override_direction")

    print(f"override: {len(ov)}/{len(df)} ({len(ov)/len(df):.1%}) | "
          f"resid sigma={sigma:.2f} notches | band coverage={coverage:.1%}")


override: 505/2029 (24.9%) | resid sigma=2.33 notches | band coverage=73.6%
